# 00 — EDA Data Dummy
**SuaraLens** | Analitik Masukan, Aduan & Aspirasi Berbasis NLP — PENS

Notebook ini melakukan eksplorasi awal dataset dummy `suaralens_dummy_simulasi.jsonl`
untuk memahami distribusi data, tren, dan pola yang relevan untuk pipeline selanjutnya.


In [ ]:
import sys
sys.path.insert(0, '..')  # akses modules dari parent

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Style global ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size':  11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/eda_summary.json'

df = pd.read_json(DATA_PATH, lines=True)
df['tanggal_masuk']   = pd.to_datetime(df['tanggal_masuk'])
df['tanggal_selesai'] = pd.to_datetime(df['tanggal_selesai'], errors='coerce')

print(f'Dataset dimuat: {len(df):,} baris, {df.shape[1]} kolom')
print(f'Rentang tanggal: {df["tanggal_masuk"].min().date()} s.d. {df["tanggal_masuk"].max().date()}')


## 1. Info Umum & Missing Values

In [ ]:
print('=== Info Dataset ===')
print(df.info())
print()

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Persen (%)': missing_pct})
print('=== Missing Values ===')
display(missing_df[missing_df['Missing'] > 0])


## 2. Distribusi Kategori

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cat_counts = df['kategori_true'].value_counts()
bars = ax.barh(cat_counts.index, cat_counts.values, color=sns.color_palette('muted', len(cat_counts)))
ax.set_xlabel('Jumlah Masukan')
ax.set_title('Distribusi Kategori Masukan')
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Distribusi Urgency & Sentimen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Urgency
urgency_order = ['Low', 'Medium', 'High', 'Critical']
urgency_counts = df['urgency_label_true'].value_counts().reindex(urgency_order)
colors_urg = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[0].bar(urgency_counts.index, urgency_counts.values, color=colors_urg)
axes[0].set_title('Distribusi Urgency')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(urgency_counts.values):
    axes[0].text(i, v + 15, f'{v:,}', ha='center', fontsize=9)

# Sentimen
sent_order = ['positive', 'neutral', 'negative']
sent_counts = df['sentiment_true'].value_counts().reindex(sent_order)
colors_sent = ['#27ae60', '#95a5a6', '#c0392b']
axes[1].bar(sent_counts.index, sent_counts.values, color=colors_sent)
axes[1].set_title('Distribusi Sentimen')
axes[1].set_ylabel('Jumlah')
for i, v in enumerate(sent_counts.values):
    axes[1].text(i, v + 15, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()


## 4. Status & SLA Breach

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Status
status_counts = df['status'].value_counts()
axes[0].pie(status_counts.values, labels=status_counts.index,
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('pastel'))
axes[0].set_title('Distribusi Status')

# SLA Breach
breach_counts = df['sla_breach_true'].value_counts()
labels = ['Tidak Breach', 'Breach'] if False in breach_counts.index else breach_counts.index
axes[1].pie(breach_counts.values,
            labels=['Tidak Breach' if not k else 'Breach' for k in breach_counts.index],
            autopct='%1.1f%%', startangle=90,
            colors=['#27ae60', '#e74c3c'])
axes[1].set_title(f'Proporsi SLA Breach (Total Breach: {df["sla_breach_true"].sum():,})')

plt.tight_layout()
plt.show()

print(f'SLA Breach rate: {df["sla_breach_true"].mean()*100:.2f}%')


## 5. Tren Bulanan (Top 3 Kategori)

In [ ]:
df['bulan'] = df['tanggal_masuk'].dt.to_period('M')
top3_cats = df['kategori_true'].value_counts().nlargest(3).index.tolist()
trend_data = df[df['kategori_true'].isin(top3_cats)]
monthly = trend_data.groupby(['bulan', 'kategori_true']).size().reset_index(name='jumlah')
monthly['bulan_str'] = monthly['bulan'].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
for cat in top3_cats:
    subset = monthly[monthly['kategori_true'] == cat]
    ax.plot(subset['bulan_str'], subset['jumlah'], marker='o', label=cat, linewidth=2)

ax.set_title('Tren Jumlah Masukan per Bulan (Top 3 Kategori)')
ax.set_xlabel('Bulan')
ax.set_ylabel('Jumlah Masukan')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


## 6. Distribusi Kanal & Stakeholder

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Kanal
kanal_counts = df['kanal'].fillna('Tidak Diketahui').value_counts()
axes[0].barh(kanal_counts.index, kanal_counts.values,
             color=sns.color_palette('Set2', len(kanal_counts)))
axes[0].set_title('Distribusi Kanal Masukan')
axes[0].set_xlabel('Jumlah')
axes[0].invert_yaxis()

# Stakeholder
sh_counts = df['stakeholder_type'].value_counts()
axes[1].bar(sh_counts.index, sh_counts.values,
            color=sns.color_palette('Set3', len(sh_counts)))
axes[1].set_title('Distribusi Stakeholder Type')
axes[1].set_ylabel('Jumlah')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 7. Distribusi Panjang Teks

In [ ]:
df['panjang_kata'] = df['teks_aduan'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['panjang_kata'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['panjang_kata'].median(), color='red', linestyle='--',
           label=f'Median: {df["panjang_kata"].median():.0f} kata')
ax.axvline(df['panjang_kata'].mean(), color='orange', linestyle='--',
           label=f'Mean: {df["panjang_kata"].mean():.1f} kata')
ax.set_title('Distribusi Panjang Teks Aduan (Jumlah Kata)')
ax.set_xlabel('Jumlah Kata')
ax.set_ylabel('Frekuensi')
ax.legend()
plt.tight_layout()
plt.show()

print(df['panjang_kata'].describe().round(1))


## 8. Cross-tab Kategori × Stakeholder (Heatmap)

In [ ]:
ct = pd.crosstab(df['kategori_true'], df['stakeholder_type'])
# Normalisasi per baris (% per kategori)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '%'})
ax.set_title('Distribusi Stakeholder per Kategori (% per baris)')
ax.set_xlabel('Stakeholder Type')
ax.set_ylabel('Kategori')
plt.tight_layout()
plt.show()


## 9. Simpan Ringkasan EDA ke JSON

In [ ]:
import json
from pathlib import Path

eda_summary = {
    'generated_at':      pd.Timestamp.now().isoformat(),
    'total_records':     len(df),
    'date_range': {
        'start': df['tanggal_masuk'].min().strftime('%Y-%m-%d'),
        'end':   df['tanggal_masuk'].max().strftime('%Y-%m-%d'),
    },
    'missing_values': {
        col: int(df[col].isnull().sum())
        for col in df.columns if df[col].isnull().any()
    },
    'kategori_distribution':  df['kategori_true'].value_counts().to_dict(),
    'urgency_distribution':   df['urgency_label_true'].value_counts().to_dict(),
    'sentiment_distribution': df['sentiment_true'].value_counts().to_dict(),
    'status_distribution':    df['status'].value_counts().to_dict(),
    'sla_breach_pct':         round(df['sla_breach_true'].mean() * 100, 2),
    'kanal_distribution':     df['kanal'].fillna('Tidak Diketahui').value_counts().to_dict(),
    'stakeholder_distribution': df['stakeholder_type'].value_counts().to_dict(),
    'text_length_stats': df['panjang_kata'].describe().round(1).to_dict(),
}

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(eda_summary, f, ensure_ascii=False, indent=2, default=str)

print(f'EDA summary disimpan ke: {OUTPUT_PATH}')


## Ringkasan Temuan

- **Dataset**: 5.150 baris masukan dari 6 bulan (Maret–Agustus 2026)
- **Distribusi kategori tidak seimbang**: Akademik dominan (~18%), Lainnya paling sedikit (~1%)
- **Sentimen mayoritas negatif** (~74%), sesuai ekspektasi platform pengaduan
- **SLA breach rate ~32%** — perlu perhatian khusus, terutama di kategori dengan breach tertinggi
- **Tren musiman terlihat**: lonjakan Keuangan di awal semester, Akademik saat masa ujian
- **Variasi panjang teks cukup tinggi** (beberapa kata s.d. ratusan kata) — penting untuk pipeline embedding
- **Cross-tab**: Masukan Keuangan didominasi Mahasiswa & Orang Tua; Kerjasama & Mitra lebih banyak dari Mitra eksternal
